In [6]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [7]:
Processed_Dataset = "20260925_pallas_interpolation_CLARA"

In [8]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [9]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [10]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 21680, range: 317)> Size: 55MB
array([[0.4315278 , 0.4267747 , 0.42132591, ...,        nan,        nan,
               nan],
       [0.43152896, 0.42677596, 0.42132723, ...,        nan,        nan,
               nan],
       [0.43153012, 0.42677721, 0.42132854, ...,        nan,        nan,
               nan],
       ...,
       [0.49986533, 0.49512856, 0.48977497, ...,        nan,        nan,
               nan],
       [0.49989651, 0.49515837, 0.489803  , ...,        nan,        nan,
               nan],
       [0.4999277 , 0.49518818, 0.48983103, ...,        nan,        nan,
               nan]], shape=(21680, 317))
Coordinates:
  * Time     (Time) datetime64[ns] 173kB 2026-09-25T00:54:34.313950538 ... 20...
  * range    (range) float64 3kB 119.2 149.1 178.9 ... 1.096e+04 1.1e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 21680, range: 3

In [11]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")